In [2]:
# Imports section
import pandas as pd
import io 
import requests
print(pd.__version__)
import sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

import numpy as np
from sklearn.metrics import make_scorer, r2_score


2.2.3


## Part 1. Loading the dataset

In [4]:
# Using pandas load the dataset (load remotely, not locally)
url = 'science_data_large.csv'
df =  pd.read_csv(url)
# Output the first 15 rows of the data
print(df.head(15))
# Display a summary of the table information (number of datapoints, etc.)
df.info()

    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature °C  1000 non-null   int64  
 1   Mols KCL        1000 non-null   int64  
 2   Size nm^3       1000 n

## Part 2. Splitting the dataset

In [5]:
# Take the pandas dataset and split it into our features (X) and label (y)
X = df[['Temperature °C', 'Mols KCL']]  # Features
y = df['Size nm^3']   # Label

print("Features shape:", X.shape)
print("Label shape:", y.shape)

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
X = X.dropna()
y = y.dropna()

X = X.loc[y.index] 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

print("Training set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)

Features shape: (1000, 2)
Label shape: (1000,)
Training set: (900, 2) (900,)
Test set: (100, 2) (100,)


## Part 3. Perform a Linear Regression

In [10]:
# Use sklearn to train a model on the training set
model = LinearRegression()
model.fit(X_train, y_train)


# Create a sample datapoint and predict the output of that sample with the trained model
sample_data_point = pd.DataFrame({"Temperature °C": [460], "Mols KCL":[345]}) 
predicted_size = model.predict(sample_data_point)
print("Predicted Size (nm³) for the sample data point:", predicted_size[0])


# Report on the score for that model, in your own words (markdown, not code) explain what the score means
r2_score = model.score(X_test, y_test)
print("R-squared score of the model:", r2_score)

# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX
print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)


Predicted Size (nm³) for the sample data point: 345315.66850527574
R-squared score of the model: 0.8552472077276095
Coefficients: [ 866.14641337 1032.69506649]
Intercept: -409391.47958340764


The R-squared score of this model is .855, which means that about 85% of the variance in the size of the slime can be explained by the changes in temperature and moles of KCL. A higher value means that the model fits the data really well, and with a value that is higher than 80%, this score is decent. This means that the predictions of this model is definitely better than just using the mean of the response variable. 

$h(x) = -409391.47858 + 866.14641 (\text{Temperature}) + 1032.69506 (\text{Mols KCL})$


## Part 4. Use Cross Validation

In [7]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
scores = cross_val_score(model, X, y, cv=5, scoring='r2')  # 5-fold cross-validation

# Print the cross-validation scores
print("Cross-validation R² scores:", scores)
print("Mean R² score:", scores.mean())
print("Standard deviation of R² scores:", scores.std())

# Report on their finding and their significance
#During the 5-fold cross-validation, we obtained various R² scores, which quantify the proportion of variance in the dependent variable (Size in nm³) that is explained by the independent variables (Temperature and Mols KCL).
#We can see that the R2 scores across all five folds stay relatively around 0.85, with a very small SD of 0.013. This means it's reasonably consistent across different subsets of data. The small standard deviation indicates that the model's performance is reliable and indicative of how it might perform on unseen data. This is essential in machine learning, as we aim for models that generalize well rather than fit only the training data.


Cross-validation R² scores: [0.83918826 0.87051239 0.85871066 0.87202623 0.84364641]
Mean R² score: 0.8568167899144437
Standard deviation of R² scores: 0.01346630737209602


## Part 5. Using Polynomial Regression

In [9]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
poly = PolynomialFeatures(degree=2)

X_poly = poly.fit_transform(X)  # Transform the original features
X_train_poly, X_test_poly, y_train, y_test = train_test_split(X_poly, y, test_size=0.1, random_state=42)
model_poly = LinearRegression()
model_poly.fit(X_train_poly, y_train)
scores_poly = cross_val_score(model_poly, X_poly, y, cv=5, scoring='r2')  # 5-fold cross-validation

print("Cross-validation R² scores for polynomial regression:", scores_poly)
print("Mean R² score for polynomial regression:", scores_poly.mean())
print("Standard deviation of R² scores for polynomial regression:", scores_poly.std())

# Report on the metrics and output the resultant equation as you did in Part 3.
# Coefficients for the features
coefficients = model_poly.coef_
# Intercept
intercept = model_poly.intercept_

# Display the coefficients and intercept
print("Coefficients:", coefficients)
print("Intercept:", intercept)
#This model perfectly explains all the variance in the data. It is perfectly consistent across different subsets of the data. 

Cross-validation R² scores for polynomial regression: [1. 1. 1. 1. 1.]
Mean R² score for polynomial regression: 1.0
Standard deviation of R² scores for polynomial regression: 0.0
Coefficients: [ 0.00000000e+00  1.20000000e+01 -1.27195488e-07  1.26494371e-11
  2.00000000e+00  2.85714287e-02]
Intercept: 2.0477105863392353e-05


$h(x) = 0.00002 + 12.00000 (\text{Temperature}) + (-0.00000) (\text{Mols KCL}) + 0.00000 (\text{Temperature}^2) + 2.00000 (\text{Mols KCL}^2) + 0.02857 (\text{Temperature})  (\text{Mols KCL})$
